# LeetCode 3736 — Minimum Moves to Equal Array Elements III bằng Python

Link đề: https://leetcode.com/problems/minimum-moves-to-equal-array-elements-iii/ · Độ khó: **Easy** · Chủ đề: Array, Math

Skeleton LeetCode:

```python
class Solution:
    def minMoves(self, nums: List[int]) -> int:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua WebSearch / leetcode.doocs.org.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi lượt được **tăng một phần tử thêm 1**; tìm số lượt ít nhất để mọi phần tử bằng nhau.
2. Nhận ra vì sao đích tốt nhất là `max(nums)` (không bao giờ giảm được phần tử, nên đích không thể nhỏ hơn max; nâng đích cao hơn chỉ tốn thêm).
3. Chuyển thành công thức `sum(max - x)` rồi rút gọn thành `n * max - sum`, dùng `max`, `min`, `sum`, `len` thành thạo.
4. Viết 5 cách khác nhau (thử mọi đích, cộng chênh lệch, công thức `n * max - sum`, sắp xếp, mảng đếm tần suất) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: dùng `min` làm đích, nhầm với bài "tăng n-1 phần tử" (LeetCode 453), chọn đích `max + 1`, và lấy trung vị (bài "giảm/tăng tùy ý" - LeetCode 462).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Trong **một lượt**, bạn được chọn **một** phần tử bất kỳ và **tăng nó thêm `1`**. Trả về **số lượt tối thiểu** để tất cả các phần tử của `nums` bằng nhau.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[2,1,3]` | `3` | Tăng `2` lên `3` (1 lượt), tăng `1` lên `2` rồi `3` (2 lượt): tổng `3` lượt, tất cả bằng `3` |
| 2 | `[4,4,5]` | `2` | Tăng hai số `4` lên `5`: `2` lượt |

Constraints: `1 <= nums.length <= 100`, `1 <= nums[i] <= 100`.

In [1]:
for nums in [[2, 1, 3], [4, 4, 5]]:
    mx = max(nums)
    diffs = [mx - x for x in nums]
    print(f"nums={nums}  đích={mx}  chênh lệch={diffs}  tổng lượt={sum(diffs)}")

nums=[2, 1, 3]  đích=3  chênh lệch=[1, 2, 0]  tổng lượt=3
nums=[4, 4, 5]  đích=5  chênh lệch=[1, 1, 0]  tổng lượt=2


## 3. Kiến thức nền

### 3.1. `max`, `min`, `sum`, `len`

Bốn hàm dựng sẵn quét list một lần: `max(a)`, `min(a)`, `sum(a)`, `len(a)`. Chúng là "công cụ chính" của bài này. `len` là `O(1)`, còn ba hàm kia là `O(n)`.

In [2]:
nums = [2, 1, 3]
print(max(nums), min(nums), sum(nums), len(nums))
print(max(nums) * len(nums) - sum(nums))

3 1 6 3
3


### 3.2. List comprehension và `sum`

`[mx - x for x in nums]` tạo list các độ chênh lệch; `sum(mx - x for x in nums)` (không có `[]`) dùng generator nên không tạo list trung gian.

In [3]:
nums = [4, 4, 5]
mx = max(nums)
print([mx - x for x in nums])
print(sum(mx - x for x in nums))

[1, 1, 0]
2


### 3.3. Công thức "tổng chênh lệch" = `n * max - sum`

Vì `sum(mx - x for x in nums) = n * mx - sum(nums)` (có `n` số hạng `mx`, rồi trừ tổng các `x`). Đây là cách tính `O(n)` chỉ với hai hàm dựng sẵn.

In [4]:
nums = [2, 1, 3]
n, mx, s = len(nums), max(nums), sum(nums)
print(n * mx, s, n * mx - s)
print(sum(mx - x for x in nums) == n * mx - s)

9 6 3
True


### 3.4. `sorted` và chỉ số cuối `[-1]`

`sorted(nums)` trả list tăng dần; `a[-1]` là phần tử lớn nhất, `a[0]` nhỏ nhất. Sau khi sắp xếp, nếu duyệt từ nhỏ đến lớn, đích luôn là phần tử cuối.

In [5]:
a = sorted([2, 1, 3])
print(a, a[0], a[-1])
print(sum(a[-1] - x for x in a))

[1, 2, 3] 1 3
3


### 3.5. Mảng đếm / `Counter`

Vì `nums[i] <= 100`, ta có thể gom các phần tử bằng nhau: giá trị `v` xuất hiện `c` lần cần `(mx - v) * c` lượt. `Counter` làm việc này gọn.

In [6]:
from collections import Counter
c = Counter([4, 4, 5])
mx = 5
print(c, sum((mx - v) * k for v, k in c.items()))

Counter({4: 2, 5: 1}) 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Mỗi lượt chỉ có thể **tăng**, nên không phần tử nào giảm được. Vì thế, đích chung `T` phải thỏa `T >= max(nums)`: phần tử lớn nhất không thể hạ xuống.

Nếu `T = max(nums)` thì phần tử `x` cần đúng `T - x` lượt. Nếu chọn `T` lớn hơn, **mọi** phần tử cần thêm lượt nên tổng tăng thêm `n * (T - max)`. Vậy đích tốt nhất là `T = max(nums)`.

Tính tay `[2, 1, 3]`: đích `3`; cần `1 + 2 + 0 = 3` lượt. Tính tay `[4, 4, 5]`: đích `5`; cần `1 + 1 + 0 = 2`.

### Bảng tính tay

| nums | max | chênh lệch mỗi phần tử | tổng | `n*max - sum` |
|---|---|---|---|---|
| `[2,1,3]` | 3 | 1, 2, 0 | 3 | 9 - 6 = 3 |
| `[4,4,5]` | 5 | 1, 1, 0 | 2 | 15 - 13 = 2 |
| `[7]` | 7 | 0 | 0 | 7 - 7 = 0 |
| `[1,100]` | 100 | 99, 0 | 99 | 200 - 101 = 99 |

In [7]:
for nums in [[2, 1, 3], [4, 4, 5], [7], [1, 100]]:
    mx = max(nums)
    print(nums, "max=", mx, [mx - x for x in nums], sum(mx - x for x in nums), len(nums) * mx - sum(nums))

[2, 1, 3] max= 3 [1, 2, 0] 3 3
[4, 4, 5] max= 5 [1, 1, 0] 2 2
[7] max= 7 [0] 0 0
[1, 100] max= 100 [99, 0] 99 99


## 5. Cách 1 — Brute Force / baseline

**Thử mọi đích** `T` từ `max(nums)` đến một cận trên (ví dụ `max(nums) + 100`), tính chi phí `sum(T - x)` của mỗi đích, lấy nhỏ nhất. Không cần "biết trước" đích là `max`; code tự tìm và ta quan sát nó rơi đúng vào `max`.

Nút thắt: ta phải thử rất nhiều đích và mỗi đích tốn `O(n)`; thực tế chỉ có **một** đích đáng xét.

In [8]:
class SolutionBrute:
    def minMoves(self, nums):
        best = None
        lo = max(nums)                         # đích không thể nhỏ hơn max (chỉ tăng được)
        for target in range(lo, lo + 101):
            cost = sum(target - x for x in nums)
            if best is None or cost < best:
                best = cost
        return best

print(SolutionBrute().minMoves([2, 1, 3]), SolutionBrute().minMoves([4, 4, 5]), SolutionBrute().minMoves([7]))

3 2 0


**Độ phức tạp:** Time `O(n * R)` với `R` là số đích thử (ở đây 101), Space `O(1)`.

In [9]:
# Quan sát: chi phí theo đích T của [2, 1, 3] — nhỏ nhất ở T = max = 3
nums = [2, 1, 3]
for T in range(3, 8):
    print(f"đích {T}: chi phí {sum(T - x for x in nums)}")
assert [sum(T - x for x in nums) for T in range(3, 8)] == [3, 6, 9, 12, 15]

đích 3: chi phí 3
đích 4: chi phí 6
đích 5: chi phí 9
đích 6: chi phí 12
đích 7: chi phí 15


## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** đích tối ưu **cố định** là `max(nums)`. Chi phí của đích `T` là `n * T - sum(nums)`, **tăng đúng `n`** mỗi khi `T` tăng thêm 1, nên đạt nhỏ nhất ở `T` nhỏ nhất hợp lệ, tức `T = max`.

**Quy tắc "nếu… thì…":**

- Nếu phần tử bằng `max` -> cần `0` lượt.
- Nếu phần tử `x < max` -> cần đúng `max - x` lượt (không ít hơn vì mỗi lượt tăng 1; không cần nhiều hơn).
- Nếu mọi phần tử đã bằng nhau -> đáp án `0`.
- Đáp án `= n * max - sum(nums)`.

**Invariant:** sau khi nâng phần tử `x` lên `max`, nó không cần động tới nữa; các phần tử độc lập với nhau vì mỗi lượt chỉ chạm một phần tử.

In [10]:
# Kiểm chứng: đích tối ưu luôn là max (so sánh với thử mọi đích trong khoảng rộng)
import random
random.seed(3736)
for _ in range(1000):
    n = random.randint(1, 100)
    nums = [random.randint(1, 100) for _ in range(n)]
    costs = {T: sum(T - x for x in nums) for T in range(max(nums), 201)}
    assert min(costs, key=costs.get) == max(nums)
    assert costs[max(nums)] == n * max(nums) - sum(nums)
print("đích tối ưu = max, chi phí = n*max - sum: đúng")

đích tối ưu = max, chi phí = n*max - sum: đúng


## 7. Các cách tiếp theo

### Cách 2 — Cộng chênh lệch từng phần tử

Tìm `mx = max(nums)`, rồi `sum(mx - x for x in nums)`. Đây là bản dịch thẳng của lập luận ở mục 4.

In [11]:
class SolutionDiff:
    def minMoves(self, nums):
        mx = max(nums)
        return sum(mx - x for x in nums)

print(SolutionDiff().minMoves([2, 1, 3]), SolutionDiff().minMoves([4, 4, 5]), SolutionDiff().minMoves([7]))

3 2 0


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 3 — Công thức `n * max - sum`

Hai hàm dựng sẵn và một phép nhân, một dòng. Đây là lời giải thường được nộp.

In [12]:
class SolutionFormula:
    def minMoves(self, nums):
        return len(nums) * max(nums) - sum(nums)

print(SolutionFormula().minMoves([2, 1, 3]), SolutionFormula().minMoves([4, 4, 5]), SolutionFormula().minMoves([7]))

3 2 0


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 4 — Sắp xếp rồi cộng dồn

Sắp xếp để phần tử cuối là `max`; sau đó cộng `a[-1] - x` cho mọi `x`. Chậm hơn (`O(n log n)`) nhưng hữu ích khi bài toán mở rộng cần thứ tự.

In [13]:
class SolutionSorted:
    def minMoves(self, nums):
        a = sorted(nums)
        top = a[-1]
        moves = 0
        for x in a:
            moves += top - x
        return moves

print(SolutionSorted().minMoves([2, 1, 3]), SolutionSorted().minMoves([4, 4, 5]), SolutionSorted().minMoves([7]))

3 2 0


**Độ phức tạp:** Time `O(n log n)`, Space `O(n)`.

### Cách 5 — Đếm tần suất

Gom các giá trị bằng nhau bằng `Counter`: mỗi giá trị `v` xuất hiện `c` lần cần `(mx - v) * c` lượt. Tốt khi có rất nhiều phần tử trùng nhau.

In [14]:
from collections import Counter

class SolutionCounter:
    def minMoves(self, nums):
        cnt = Counter(nums)
        mx = max(cnt)
        return sum((mx - v) * c for v, c in cnt.items())

print(SolutionCounter().minMoves([2, 1, 3]), SolutionCounter().minMoves([4, 4, 5]), SolutionCounter().minMoves([7]))

3 2 0


**Độ phức tạp:** Time `O(n)`, Space `O(d)` với `d` là số giá trị khác nhau.

## 8. TRACE từng bước

Trace Cách 2 (cộng chênh lệch) với `nums = [2, 1, 3, 3, 1]`:

| bước | x | `mx - x` | tổng dồn |
|---|---|---|---|
| 1 | 2 | 1 | 1 |
| 2 | 1 | 2 | 3 |
| 3 | 3 | 0 | 3 |
| 4 | 3 | 0 | 3 |
| 5 | 1 | 2 | 5 |

Kiểm tra công thức: `n * max - sum = 5 * 3 - 10 = 5`.

In [15]:
def trace_diff(nums):
    mx = max(nums)
    print(f"nums={nums}  max={mx}")
    total = 0
    for step, x in enumerate(nums, 1):
        need = mx - x
        total += need
        print(f"  bước {step}: x={x} cần {need} lượt -> tổng dồn {total}")
    print(f"  công thức n*max-sum = {len(nums)}*{mx}-{sum(nums)} = {len(nums) * mx - sum(nums)}")
    return total

assert trace_diff([2, 1, 3, 3, 1]) == 5
assert trace_diff([4, 4, 5]) == 2

nums=[2, 1, 3, 3, 1]  max=3
  bước 1: x=2 cần 1 lượt -> tổng dồn 1
  bước 2: x=1 cần 2 lượt -> tổng dồn 3
  bước 3: x=3 cần 0 lượt -> tổng dồn 3
  bước 4: x=3 cần 0 lượt -> tổng dồn 3
  bước 5: x=1 cần 2 lượt -> tổng dồn 5
  công thức n*max-sum = 5*3-10 = 5
nums=[4, 4, 5]  max=5
  bước 1: x=4 cần 1 lượt -> tổng dồn 1
  bước 2: x=4 cần 1 lượt -> tổng dồn 2
  bước 3: x=5 cần 0 lượt -> tổng dồn 2
  công thức n*max-sum = 3*5-13 = 2


In [16]:
def trace_target(nums):
    print("thử đích:", nums)
    best = None
    for T in range(max(nums), max(nums) + 4):
        cost = sum(T - x for x in nums)
        mark = ""
        if best is None or cost < best:
            best, mark = cost, " <- tốt nhất tới giờ"
        print(f"  T={T}: chi phí={cost}{mark}")
    return best

assert trace_target([2, 1, 3]) == 3

thử đích: [2, 1, 3]
  T=3: chi phí=3 <- tốt nhất tới giờ
  T=4: chi phí=6
  T=5: chi phí=9
  T=6: chi phí=12


## 9. Code nộp LeetCode

In [17]:
from typing import List, Optional

class Solution:
    def minMoves(self, nums: List[int]) -> int:
        return len(nums) * max(nums) - sum(nums)

## 10. Vì sao thuật toán đúng

- **Cận dưới cho đích:** mỗi lượt chỉ tăng, nên giá trị cuối `T` của mọi phần tử phải `>= max(nums)`. Phần tử lớn nhất không hạ xuống được.
- **Chi phí từng phần tử độc lập:** để đưa `x` lên `T` cần **đúng** `T - x` lượt (mỗi lượt cộng 1). Các phần tử không ảnh hưởng nhau vì một lượt chỉ chạm một phần tử, nên tổng chi phí là `sum(T - x) = n * T - sum(nums)`.
- **Đích tối ưu:** hàm chi phí `n * T - sum(nums)` tăng theo `T` (hệ số `n >= 1`), nên nhỏ nhất ở `T` nhỏ nhất cho phép, `T = max(nums)`.
- **Đạt được:** với `T = max`, mỗi `x` đúng `max - x >= 0` lượt, nên cách chọn hợp lệ, và theo các điểm trên không thể ít hơn.
- **Biên:** `n = 1` -> `1 * x - x = 0`; mọi phần tử bằng nhau -> `0`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| Thử mọi đích | `O(n * R)` | `O(1)` |
| Cộng chênh lệch | `O(n)` | `O(1)` |
| Công thức `n * max - sum` | `O(n)` | `O(1)` |
| Sắp xếp rồi cộng | `O(n log n)` | `O(n)` |
| Đếm tần suất | `O(n)` | `O(d)` |

`n = len(nums)`, `R` là số đích thử, `d` là số giá trị khác nhau. Phải đọc mọi phần tử nên `O(n)` là tối ưu. Với `n <= 100` mọi cách đều tức thì. Đáp án tối đa `99 * 100 = 9900`, nằm trong `int` thông thường.

## 12. Bộ test

Ví dụ của đề, các biên (một phần tử, mọi phần tử bằng nhau, một phần tử nhỏ + nhiều phần tử lớn, `n = 100`, giá trị `1` và `100`) và so sánh **tất cả cách** trên input ngẫu nhiên hợp lệ.

In [18]:
impls = [SolutionBrute(), SolutionDiff(), SolutionFormula(), SolutionSorted(), SolutionCounter(), Solution()]

cases = [
    ([2, 1, 3], 3), ([4, 4, 5], 2), ([7], 0), ([1, 1, 1], 0), ([1, 100], 99), ([100, 1], 99),
    ([1] + [100] * 99, 99), ([1] * 99 + [100], 99 * 99), (list(range(1, 101)), 4950), ([100] * 100, 0),
    ([5, 5, 5, 6], 3), ([3, 2], 1),
]
for nums, exp in cases:
    assert 1 <= len(nums) <= 100 and all(1 <= v <= 100 for v in nums)
    for impl in impls:
        got = impl.minMoves(list(nums))
        assert got == exp, (type(impl).__name__, nums, got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 12 ca


In [19]:
import random
random.seed(37360)
for _ in range(800):
    n = random.randint(1, 100)
    hi = random.choice([2, 10, 100])
    nums = [random.randint(1, hi) for _ in range(n)]
    exp = sum(max(nums) - x for x in nums)
    outs = [impl.minMoves(list(nums)) for impl in impls]
    assert all(o == exp for o in outs), (nums, outs, exp)
    assert 0 <= exp <= 99 * 100
print("800 test ngẫu nhiên (seed=37360): 6 cách giống nhau")

800 test ngẫu nhiên (seed=37360): 6 cách giống nhau


In [20]:
# Mô phỏng thật: nâng dần phần tử nhỏ nhất lên 1 cho tới khi bằng nhau -> số lượt đúng bằng đáp án
for _ in range(200):
    n = random.randint(1, 12)
    nums = [random.randint(1, 20) for _ in range(n)]
    a, moves = list(nums), 0
    while len(set(a)) > 1:
        a[a.index(min(a))] += 1
        moves += 1
    assert moves == Solution().minMoves(nums) and len(set(a)) == 1 and a[0] == max(nums)
print("mô phỏng từng lượt khớp công thức, đích = max")

mô phỏng từng lượt khớp công thức, đích = max


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `min` làm đích

Chỉ được **tăng**, nên không thể đưa phần tử lớn xuống `min`. Dùng `x - min` là bài toán *giảm* phần tử.

In [21]:
def wrong_min_target(nums):
    mn = min(nums)
    return sum(x - mn for x in nums)       # SAI: đích là min nghĩa là phải giảm phần tử

print(wrong_min_target([2, 1, 3]), "vs", Solution().minMoves([2, 1, 3]))
print(wrong_min_target([4, 4, 5]), "!=", Solution().minMoves([4, 4, 5]))
assert wrong_min_target([4, 4, 5]) == 1 != 2

3 vs 3
1 != 2


### Lỗi 2: nhầm với bài "tăng n-1 phần tử" (LeetCode 453)

Ở bài 453 mỗi lượt **tăng `n - 1` phần tử** thêm 1, đáp án `sum - n * min`. Bài này tăng **một** phần tử mỗi lượt nên đáp án khác.

In [22]:
def wrong_problem_453(nums):
    return sum(nums) - len(nums) * min(nums)      # SAI: công thức bài 453

print(wrong_problem_453([4, 4, 5]), "!=", Solution().minMoves([4, 4, 5]))
assert wrong_problem_453([4, 4, 5]) == 1 != 2
assert wrong_problem_453([1, 100]) == 99 == Solution().minMoves([1, 100])   # trùng hợp ở ca hai phần tử

1 != 2


### Lỗi 3: chọn đích `max + 1` (hoặc cao hơn "cho chắc")

Đích cao hơn `max` chỉ làm mọi phần tử tốn thêm lượt: thêm đúng `n` lượt cho mỗi đơn vị.

In [23]:
def wrong_target_plus_one(nums):
    T = max(nums) + 1                       # SAI: không cần vượt max
    return sum(T - x for x in nums)

print(wrong_target_plus_one([2, 1, 3]), "!=", Solution().minMoves([2, 1, 3]))
assert wrong_target_plus_one([2, 1, 3]) == 3 + 3 != 3

6 != 3


### Lỗi 4: dùng trung vị (nhầm với bài "tăng/giảm tuỳ ý")

Khi cho phép cả tăng lẫn giảm (LeetCode 462), đích tối ưu là **trung vị**. Ở đây chỉ được tăng, nên trung vị thường **không hợp lệ**.

In [24]:
def wrong_median(nums):
    a = sorted(nums)
    med = a[len(a) // 2]
    return sum(abs(x - med) for x in a)     # SAI: cho phép giảm phần tử

print(wrong_median([1, 1, 100]), "!=", Solution().minMoves([1, 1, 100]))
assert wrong_median([1, 1, 100]) == 99 != 198 == Solution().minMoves([1, 1, 100])

99 != 198


### Lỗi 5: tính cặp thay vì từng phần tử

Một số bạn cộng `abs(a - b)` trên các cặp liền kề, đếm sai khi mảng chưa sắp xếp. Mỗi phần tử phải so với **đích chung** `max`, không phải với hàng xóm.

In [25]:
def wrong_adjacent(nums):
    return sum(abs(nums[i] - nums[i + 1]) for i in range(len(nums) - 1))     # SAI

print(wrong_adjacent([5, 1, 5]), "!=", Solution().minMoves([5, 1, 5]))
assert wrong_adjacent([5, 1, 5]) == 8 and Solution().minMoves([5, 1, 5]) == 4 and wrong_adjacent([5, 1, 5]) != 4

8 != 4


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mỗi lượt tăng một phần tử thêm 1, làm bằng nhau" -> đích `= max`, đáp án `n * max - sum`.
- "Mỗi lượt giảm một phần tử" -> đích `= min`, đáp án `sum - n * min`.
- "Tăng hoặc giảm tuỳ ý" -> đích là **trung vị**.
- "Mỗi lượt tăng `n-1` phần tử" (453) -> tương đương giảm 1 phần tử: `sum - n * min`.
- Luôn tự hỏi: *một lượt đụng tới mấy phần tử và theo hướng nào?* — đó là yếu tố quyết định công thức.
- Kiểm tra nhanh bằng ví dụ 2 `[4,4,5] -> 2`: các công thức nhầm thường cho `1`.

### Đối chiếu bốn "luật chơi" bằng BFS trên mảng nhỏ

Để tin rằng công thức phụ thuộc vào **luật một lượt**, ta dùng BFS (tìm kiếm theo tầng) trên các mảng rất nhỏ và so với công thức: (a) tăng 1 phần tử, (b) giảm 1 phần tử, (c) tăng `n-1` phần tử.

In [26]:
from collections import deque

def bfs_min(nums, moves, limit=12):
    start = tuple(nums)
    seen = {start}
    q = deque([(start, 0)])
    while q:
        state, d = q.popleft()
        if len(set(state)) == 1:
            return d
        for nxt in moves(state):
            if max(nxt) <= limit and min(nxt) >= 0 and nxt not in seen:
                seen.add(nxt)
                q.append((nxt, d + 1))
    return -1

inc_one = lambda s: [s[:i] + (s[i] + 1,) + s[i + 1:] for i in range(len(s))]
dec_one = lambda s: [s[:i] + (s[i] - 1,) + s[i + 1:] for i in range(len(s))]
inc_rest = lambda s: [tuple(v + (j != i) for j, v in enumerate(s)) for i in range(len(s))]

for nums in [[2, 1, 3], [4, 4, 5], [1, 3], [1, 2, 2]]:
    a = bfs_min(nums, inc_one)
    b = bfs_min(nums, dec_one)
    c = bfs_min(nums, inc_rest)
    print(nums, "tăng 1:", a, "| giảm 1:", b, "| tăng n-1:", c)
    assert a == len(nums) * max(nums) - sum(nums)
    assert b == sum(nums) - len(nums) * min(nums)
    assert c == sum(nums) - len(nums) * min(nums)

[2, 1, 3] tăng 1: 3 | giảm 1: 3 | tăng n-1: 3
[4, 4, 5] tăng 1: 2 | giảm 1: 1 | tăng n-1: 1
[1, 3] tăng 1: 2 | giảm 1: 2 | tăng n-1: 2
[1, 2, 2] tăng 1: 1 | giảm 1: 2 | tăng n-1: 2


## 15. Biến thể của bài

1. **Minimum Moves to Equal Array Elements** (LeetCode 453): mỗi lượt tăng `n - 1` phần tử thêm 1.
2. **Minimum Moves to Equal Array Elements II** (462): mỗi lượt tăng hoặc giảm 1 phần tử; đích là trung vị.
3. **Minimum Moves to Equal Array Elements** với giới hạn: mỗi phần tử chỉ được tăng tối đa `k` lần.
4. **Make Array Elements Equal** với lượt thay đổi `x` thay vì `1`: đáp án `(max - x) / step` nếu chia hết.
5. **Minimum Operations to Make Array Equal** (1551): mảng `2i + 1`, mỗi lượt chuyển 1 đơn vị giữa hai phần tử.
6. **Minimum Cost to Make Array Equal** (2448): mỗi phần tử có chi phí riêng; đích là trung vị có trọng số.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Thử mọi đích | tính chi phí cho từng `T` | `O(n * R)` | `O(1)` | Oracle kiểm tra, khám phá |
| Cộng chênh lệch | `sum(max - x)` | `O(n)` | `O(1)` | Dễ hiểu nhất |
| Công thức | `n * max - sum` | `O(n)` | `O(1)` | Nộp bài, một dòng |
| Sắp xếp | `a[-1] - x` trên list đã sắp xếp | `O(n log n)` | `O(n)` | Khi bài mở rộng cần thứ tự |
| Đếm tần suất | `(max - v) * c` | `O(n)` | `O(d)` | Nhiều phần tử trùng |

Nộp công thức `n * max - sum`.

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().minMoves([3, 3, 3])` và `Solution().minMoves([1, 2, 3, 4])` ra gì?

**Bài 2 (tính tay):** với `nums = [5, 1, 5, 2]`, tính đích và số lượt tối thiểu.

**Bài 3 (điền TODO):** hoàn thành `min_moves_decrease(nums)`: mỗi lượt **giảm** một phần tử đi 1 (được phép giảm xuống `0` hoặc âm), tìm số lượt tối thiểu để mọi phần tử bằng nhau.

**Bài 4 (tự viết lại):** viết `min_moves_step(nums, step)`: mỗi lượt tăng một phần tử thêm `step`. Trả `-1` nếu không thể làm bằng nhau.

**Bài 5 (mini challenge):** viết `min_moves_median(nums)` cho phép **tăng hoặc giảm** 1 mỗi lượt (LeetCode 462) rồi so sánh với `minMoves` để thấy chúng khác nhau.

In [27]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().minMoves([3, 3, 3]), Solution().minMoves([1, 2, 3, 4]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def min_moves_decrease_todo(nums):
    # TODO: đích là min(nums); đáp án = tổng (x - min)
    return 0

0 6


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: mảng toàn bằng nhau thì chưa cần lượt nào. Với `[1,2,3,4]` đích là `4`. Bài 2: đích `5`.

**Mức 2 (vừa):** Bài 3: bây giờ chỉ được giảm, nên đích là `min`. Bài 4: mỗi `max - x` phải chia hết cho `step`; nếu không thì `-1`. Bài 5: sắp xếp và dùng phần tử giữa làm đích.

**Mức 3 (gần đáp án):** Bài 1: `0` và `3 + 2 + 1 + 0 = 6`. Bài 2: `0 + 4 + 0 + 3 = 7`. Bài 4: `min_moves_step([1, 3, 5], 2) = (5-1)/2 + (5-3)/2 + 0 = 3`; `min_moves_step([1, 2], 2) = -1`.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1, 2
assert Solution().minMoves([3, 3, 3]) == 0 and Solution().minMoves([1, 2, 3, 4]) == 6
assert Solution().minMoves([5, 1, 5, 2]) == 7

# Bài 3
def min_moves_decrease(nums):
    mn = min(nums)
    return sum(x - mn for x in nums)
assert min_moves_decrease([2, 1, 3]) == 3 and min_moves_decrease([4, 4, 5]) == 1 and min_moves_decrease([7]) == 0

# Bài 4
def min_moves_step(nums, step):
    mx = max(nums)
    total = 0
    for x in nums:
        if (mx - x) % step != 0:
            return -1
        total += (mx - x) // step
    return total
assert min_moves_step([1, 3, 5], 2) == 3 and min_moves_step([1, 2], 2) == -1 and min_moves_step([4, 4], 7) == 0
assert min_moves_step([2, 1, 3], 1) == Solution().minMoves([2, 1, 3])

# Bài 5
def min_moves_median(nums):
    a = sorted(nums)
    med = a[len(a) // 2]
    return sum(abs(x - med) for x in a)
assert min_moves_median([1, 2, 3]) == 2 and Solution().minMoves([1, 2, 3]) == 3
assert min_moves_median([1, 1, 100]) < Solution().minMoves([1, 1, 100])
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "làm bằng nhau bằng cách chỉ tăng":**

```python
return len(nums) * max(nums) - sum(nums)
# đích = max; mỗi phần tử x cần (max - x) lượt
```

**Checklist trước khi submit:**

- [ ] Mỗi lượt đụng bao nhiêu phần tử, theo hướng nào (chỉ tăng / chỉ giảm / cả hai)?
- [ ] Chỉ tăng một phần tử -> đích `max`, không phải `min`, trung vị hay `max + 1`?
- [ ] Công thức `n * max - sum` (không phải `sum - n * min`)?
- [ ] Mảng toàn bằng nhau hoặc `n = 1` -> trả `0`?
- [ ] Thử `[2,1,3] -> 3` và `[4,4,5] -> 2`?
- [ ] Chữ ký đúng `minMoves(self, nums: List[int]) -> int`?